<a href="https://colab.research.google.com/github/bono-p/tardigrade/blob/main/notebooks/finetune_nllb_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning NLLB-200 pour la traduction Français ↔ Fulfulde (Adamawa, Cameroun)

**Projet de démonstration de faisabilité** — corpus biblique (12 000 paires de phrases alignées).

Ce notebook fait tout le pipeline :
1. Connexion à Google Drive (sauvegarde + reprise automatique après interruption)
2. Validation des données (non bloquante)
3. Split train/val/test
4. Fine-tuning bidirectionnel (FR→FF et FF→FR) de `facebook/nllb-200-distilled-600M`
5. Évaluation (BLEU / chrF++)
6. Sauvegarde finale + démo d'inférence

⚠️ **Limites à garder en tête** (voir README du dépôt pour plus de détails) :
- NLLB-200 ne contient pas de code dédié pour le fulfulde Adamawa Cameroun spécifiquement —
  on réutilise le code `fuv_Latn` (Nigerian Fulfulde), la variété la plus proche disponible.
  Le fine-tuning va adapter ce code vers ton dialecte, mais une validation par locuteur natif
  reste indispensable.
- Le corpus est 100% biblique : le modèle sera bon sur ce registre, moins fiable hors domaine.
- Ce notebook est conçu pour un GPU Colab gratuit (T4). Les temps indiqués sont approximatifs.


## 1. Connexion à Google Drive et configuration des chemins

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === Dossier racine du projet sur ton Drive ===
# Les checkpoints sont sauvegardés sur Drive, mais seulement
# 1 fois toutes les 5 évaluations (save_steps=500) pour limiter
# l'espace utilisé et les écritures Drive.
PROJECT_DIR = "/content/drive/MyDrive/fr-fulfulde-mt"

DATA_RAW_DIR       = os.path.join(PROJECT_DIR, "data/raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_DIR, "data/processed")
REPORTS_DIR        = os.path.join(PROJECT_DIR, "reports")
CHECKPOINTS_DIR    = os.path.join(PROJECT_DIR, "checkpoints")
FINAL_MODEL_DIR    = os.path.join(PROJECT_DIR, "model_final")

for d in [DATA_RAW_DIR, DATA_PROCESSED_DIR, REPORTS_DIR, CHECKPOINTS_DIR, FINAL_MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Arborescence créée sous :", PROJECT_DIR)
print("   → Checkpoints sauvegardés sur Drive 1 fois / 5 évaluations (500 steps)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Arborescence créée sous : /content/drive/MyDrive/fr-fulfulde-mt
   → Checkpoints sauvegardés sur Drive 1 fois / 5 évaluations (500 steps)


## 2. Dépôt du corpus

Dépose ton fichier `corpus.csv` (colonnes `french_text`, `fulfulde_text`) dans :
`Mon Drive/fr-fulfulde-mt/data/raw/corpus.csv`

Tu peux le faire via l'interface Drive avant de lancer ce notebook, ou décommenter la cellule
suivante pour l'uploader directement depuis ton ordinateur.


In [ ]:
# Option : upload direct depuis ton ordinateur (décommente si tu n'as pas déjà déposé le fichier)
# from google.colab import files
# uploaded = files.upload()
# import shutil
# for fname in uploaded.keys():
#     shutil.move(fname, os.path.join(DATA_RAW_DIR, "corpus.csv"))

RAW_CSV_PATH = os.path.join(DATA_RAW_DIR, "corpus.csv")
assert os.path.exists(RAW_CSV_PATH), f"Fichier introuvable: {RAW_CSV_PATH} -- dépose-le d'abord."
print("Corpus trouvé:", RAW_CSV_PATH)


Corpus trouvé: /content/drive/MyDrive/fr-fulfulde-mt/data/raw/corpus.csv


## 3. Validation des données (non bloquante)

On vérifie l'encodage, les doublons, les lignes vides, les ratios de longueur suspects et les
caractères inattendus. **Rien n'est supprimé automatiquement** sauf les lignes 100% vides
(inutilisables des deux côtés). Le rapport complet est sauvegardé sur Drive pour que tu
l'examines à ton rythme — l'entraînement n'attend pas une correction manuelle pour démarrer.


In [ ]:
!pip install -q pandas

import sys
sys.path.insert(0, "/content")

# On récupère le script de validation depuis le dépôt cloné (voir cellule suivante)
# ou on le redéfinit inline si le dépôt n'est pas cloné dans cette session.


In [ ]:
# Clone du dépôt GitHub (remplace par l'URL de ton dépôt une fois créé)
# Si tu préfères travailler sans clone, les scripts sont aussi recopiés inline plus bas.
REPO_URL = "https://github.com/bono-p/tardigrade.git"  # <-- à adapter

if not os.path.exists("/content/fr-fulfulde-mt"):
    !git clone {REPO_URL} /content/fr-fulfulde-mt 2>/dev/null || echo "Clone échoué ou dépôt non encore public -- on utilise les scripts inline."

SCRIPTS_DIR = "/content/fr-fulfulde-mt/scripts" if os.path.exists("/content/fr-fulfulde-mt/scripts") else None
print("Scripts dir:", SCRIPTS_DIR)


Scripts dir: /content/fr-fulfulde-mt/scripts


In [ ]:
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

CLEAN_CSV_PATH = os.path.join(DATA_PROCESSED_DIR, "corpus_clean.csv")
REPORT_PATH = os.path.join(REPORTS_DIR, "validation_report.txt")

if SCRIPTS_DIR:
    validate_mod = load_module("validate_data", os.path.join(SCRIPTS_DIR, "validate_data.py"))
    df = validate_mod.load_csv(__import__("pathlib").Path(RAW_CSV_PATH))
    report = validate_mod.run_validation(df)
    validate_mod.write_report(report, __import__("pathlib").Path(REPORT_PATH))
    df_clean = df.drop(index=report["rows_to_drop"]).reset_index(drop=True)
    df_clean.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8")
    print(f"{len(df_clean)} lignes conservées sur {len(df)}.")
else:
    raise RuntimeError("Clone le dépôt d'abord (cellule précédente), ou copie scripts/validate_data.py manuellement sur Drive.")

print()
print(open(REPORT_PATH, encoding='utf-8').read()[:3000])


Rapport écrit dans : /content/drive/MyDrive/fr-fulfulde-mt/reports/validation_report.txt
12699 lignes conservées sur 12699.

RAPPORT DE VALIDATION - Corpus français / fulfulde
Lignes totales analysées : 12699
Lignes 100% vides (supprimées automatiquement) : 0
Seuil ratio longueur utilisé : [0.33 ; 3.0]

--- Lignes avec texte français vide (mais fulfulde présent) (0) ---
  Aucun problème détecté.

--- Lignes avec texte fulfulde vide (mais français présent) (0) ---
  Aucun problème détecté.

--- Paires dupliquées exactes (fr+ff identiques) (482) ---
  lignes 207 et 262
  lignes 335 et 357
  lignes 335 et 663
  lignes 335 et 860
  lignes 866 et 871
  lignes 866 et 888
  lignes 335 et 899
  lignes 1075 et 1099
  lignes 779 et 1143
  lignes 335 et 1175
  lignes 2782 et 2788
  lignes 2782 et 2793
  lignes 2782 et 2804
  lignes 2825 et 2829
  lignes 335 et 2844
  lignes 335 et 2857
  lignes 335 et 2873
  lignes 2796 et 2901
  lignes 2796 et 2907
  lignes 335 et 2918
  ... et 462 de plus (voir

## 4. Split train / val / test (90 / 5 / 5, seed fixe)

In [ ]:
import pandas as pd

# Note : split_data.py contient la même logique pour usage CLI.
# Ici on l'implémente directement pour la lisibilité du notebook.

df = pd.read_csv(CLEAN_CSV_PATH, encoding="utf-8")

import unicodedata as _ud
def _nfc(x): return _ud.normalize('NFC', str(x)) if not __import__('pandas').isna(x) else x
df['french_text']   = df['french_text'].apply(_nfc)
df['fulfulde_text'] = df['fulfulde_text'].apply(_nfc)
print('✅ Normalisation NFC appliquée au corpus chargé.')
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

n = len(df)
n_val   = max(1, int(n * 0.05))
n_test  = max(1, int(n * 0.05))
n_train = n - n_val - n_test

train_df = df.iloc[:n_train].reset_index(drop=True)
val_df   = df.iloc[n_train:n_train + n_val].reset_index(drop=True)
test_df  = df.iloc[n_train + n_val:].reset_index(drop=True)

os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
train_df.to_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), index=False, encoding="utf-8")
val_df.to_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"),   index=False, encoding="utf-8")
test_df.to_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), index=False, encoding="utf-8")

print(f"✅ Corpus splitté — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

✅ Corpus splitté — Train: 11431 | Val: 634 | Test: 634


## 5. Installation des dépendances ML


In [ ]:
!pip install -q transformers==4.* accelerate sentencepiece sacrebleu evaluate datasets

import torch
print("GPU disponible:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU seulement")


GPU disponible: True - Tesla T4


## 6. Chargement du modèle de base : NLLB-200-distilled-600M

On choisit le checkpoint **600M distillé** plutôt qu'un modèle plus gros : avec seulement
~12 000 paires de phrases, un modèle plus grand (1.3B, 3.3B) risquerait de sur-apprendre / d'oublier
ses capacités multilingues de base sans gain garanti. Le 600M offre le meilleur compromis
capacité / risque de surapprentissage pour ce volume de données, et tourne confortablement sur
un GPU T4 gratuit.

**Codes de langue utilisés :**
- Français : `fra_Latn`
- Fulfulde : `fuv_Latn` (seul code fulfulde existant dans NLLB-200 — couvre nativement le
  Nigerian Fulfulde, la variété la plus proche disponible du fulfulde Adamawa Cameroun).
  Le fine-tuning va spécialiser ce code vers ton dialecte à partir de tes données.


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "fra_Latn"
TGT_LANG = "fuv_Latn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Modèle chargé sur", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Modèle chargé sur cuda


## 7. Dataset bidirectionnel (FR→FF et FF→FR)

Chaque paire du corpus génère **deux exemples d'entraînement** (augmentation gratuite ×2).

**Corrections appliquées :**
- ✅ **Pré-tokenisation en `__init__`** — élimine la race condition sur `tokenizer.src_lang` (bug silencieux avec `num_workers > 0`)
- ✅ **Log du taux de troncation** — alerte si `MAX_LENGTH` est trop petit
- ✅ **Datasets unidirectionnels séparés** pour l'évaluation finale (FR→FF et FF→FR indépendants)
- ✅ `dataloader_num_workers=0` forcé dans les TrainingArguments (sécurité supplémentaire)


In [ ]:
import torch
from torch.utils.data import Dataset

MAX_LENGTH = 128  # Augmenter à 192 si le taux de troncation dépasse 10 %

class TranslationDataset(Dataset):
    """
    Dataset de traduction pré-tokenisé.

    La tokenisation est faite entièrement dans __init__ (séquentiellement),
    ce qui élimine toute race condition sur tokenizer.src_lang lors de
    l'itération avec plusieurs workers DataLoader.

    Args:
        df           : DataFrame avec colonnes 'french_text' et 'fulfulde_text'
        tokenizer    : tokenizer NLLB chargé
        src_lang     : code langue source (ex. 'fra_Latn')
        tgt_lang     : code langue cible  (ex. 'fuv_Latn')
        max_length   : longueur max de séquence en tokens
        bidirectional: si True, ajoute aussi la direction inverse (×2 exemples)
        label        : étiquette affichée dans les logs
    """
    def __init__(self, df, tokenizer, src_lang, tgt_lang,
                 max_length=MAX_LENGTH, bidirectional=True, label="dataset"):
        self.label = label

        # Construire la liste des paires (src_text, tgt_text, s_lang, t_lang)
        pairs = []
        for _, row in df.iterrows():
            fr = str(row["french_text"]).strip()
            ff = str(row["fulfulde_text"]).strip()
            if not fr or not ff:
                continue
            pairs.append((fr, ff, src_lang, tgt_lang))
            if bidirectional:
                pairs.append((ff, fr, tgt_lang, src_lang))

        if not pairs:
            raise ValueError(f"[{label}] Le DataFrame est vide ou ne contient pas de lignes valides.")

        # ── Pré-tokenisation séquentielle ──────────────────────────────
        # tokenizer.src_lang est modifié ici dans __init__, en séquentiel,
        # donc aucune concurrence possible. __getitem__ ne touche plus jamais
        # à l'état global du tokenizer.
        print(f"[{label}] Pré-tokenisation de {len(pairs)} exemples...")
        self.tokenized = []
        n_truncated = 0
        for src_text, tgt_text, s_lang, t_lang in pairs:
            tokenizer.src_lang = s_lang
            tokenizer.tgt_lang = t_lang

            src_enc = tokenizer(src_text,       max_length=max_length, truncation=True)
            tgt_enc = tokenizer(text_target=tgt_text, max_length=max_length, truncation=True)

            # Détecter les troncations (longueur == max_length = signe de coupure)
            if (len(src_enc["input_ids"]) >= max_length or
                    len(tgt_enc["input_ids"]) >= max_length):
                n_truncated += 1

            item = dict(src_enc)
            item["labels"] = tgt_enc["input_ids"]
            self.tokenized.append(item)

        pct = 100 * n_truncated / len(pairs)
        icon = "⚠️ " if pct > 10 else "✅"
        msg  = f" → envisage d'augmenter MAX_LENGTH à 192 ou 256." if pct > 10 else ""
        print(f"{icon} [{label}] {len(pairs)} exemples | Troncations : {n_truncated} ({pct:.1f}%){msg}")

    def __len__(self):
        return len(self.tokenized)

    def __getitem__(self, idx):
        # Retourne simplement l'exemple pré-tokenisé — pas de modification du tokenizer
        return self.tokenized[idx]


# ── Création des datasets ───────────────────────────────────────────────────

# Datasets bidirectionnels pour l'entraînement et la validation
train_dataset = TranslationDataset(train_df, tokenizer, SRC_LANG, TGT_LANG,
                                   bidirectional=True,  label="TRAIN")
val_dataset   = TranslationDataset(val_df,   tokenizer, SRC_LANG, TGT_LANG,
                                   bidirectional=True,  label="VAL  ")

# Datasets UNIDIRECTIONNELS pour l'évaluation finale par direction
# (permet de distinguer les performances FR→FF vs FF→FR)
test_fr2ff = TranslationDataset(test_df, tokenizer, SRC_LANG, TGT_LANG,
                                bidirectional=False, label="TEST FR→FF")
test_ff2fr = TranslationDataset(test_df, tokenizer, TGT_LANG, SRC_LANG,
                                bidirectional=False, label="TEST FF→FR")

print()
print(f"  Train    : {len(train_dataset):>5} exemples (bidirectionnel)")
print(f"  Val      : {len(val_dataset):>5} exemples (bidirectionnel)")
print(f"  Test (→) : {len(test_fr2ff):>5} exemples (FR → FF)")
print(f"  Test (←) : {len(test_ff2fr):>5} exemples (FF → FR)")

# Datasets de validation unidirectionnels — diagnostic par direction après entraînement
val_fr2ff = TranslationDataset(val_df, tokenizer, SRC_LANG, TGT_LANG,
                               bidirectional=False, label="VAL FR→FF")
val_ff2fr = TranslationDataset(val_df, tokenizer, TGT_LANG, SRC_LANG,
                               bidirectional=False, label="VAL FF→FR")
print(f"  Val  (→) : {len(val_fr2ff):>5} exemples (FR → FF, diagnostic directionnel)")
print(f"  Val  (←) : {len(val_ff2fr):>5} exemples (FF → FR, diagnostic directionnel)")

[TRAIN] Pré-tokenisation de 22862 exemples...
✅ [TRAIN] 22862 exemples | Troncations : 4 (0.0%)
[VAL  ] Pré-tokenisation de 1268 exemples...
✅ [VAL  ] 1268 exemples | Troncations : 0 (0.0%)
[TEST FR→FF] Pré-tokenisation de 634 exemples...
✅ [TEST FR→FF] 634 exemples | Troncations : 0 (0.0%)
[TEST FF→FR] Pré-tokenisation de 634 exemples...
✅ [TEST FF→FR] 634 exemples | Troncations : 0 (0.0%)

  Train    : 22862 exemples (bidirectionnel)
  Val      :  1268 exemples (bidirectionnel)
  Test (→) :   634 exemples (FR → FF)
  Test (←) :   634 exemples (FF → FR)


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)


## 8. Configuration de l'entraînement avec checkpointing Drive + reprise automatique

Les checkpoints sont écrits directement sur Drive (`CHECKPOINTS_DIR`). Si Colab se déconnecte
ou si tu relances ce notebook plus tard, la cellule ci-dessous **détecte automatiquement** le
dernier checkpoint et reprend l'entraînement à partir de là, sans rien perdre.


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers.trainer_utils import get_last_checkpoint

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINTS_DIR,

    # ── Évaluation & sauvegarde ──────────────────────────────────────
    eval_strategy="steps",
    eval_steps=100,           # Évaluation toutes les 100 steps
    save_strategy="steps",
    save_steps=500,           # Sauvegarde 1 fois sur 5 évaluations (100 × 5)
                              # Réduit l'I/O disque sans perdre en sécurité
    save_total_limit=2,       # Garde uniquement les 2 derniers checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ── Optimiseur ───────────────────────────────────────────────────
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,    # Batch effectif = 2 × 8 = 16
    num_train_epochs=15,
    warmup_ratio=0.05,
    weight_decay=0.01,

    # ── Génération ───────────────────────────────────────────────────
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=5,           # Cohérence avec la démo d'inférence

    # ── Mémoire / performance ─────────────────────────────────────────
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    dataloader_num_workers=0,         # Sécurité race condition tokenizer

    # ── Logs ─────────────────────────────────────────────────────────
    logging_steps=20,
    report_to="none",
)

# Détection automatique d'un checkpoint local pour reprendre l'entraînement
last_checkpoint = None
if os.path.isdir(CHECKPOINTS_DIR) and os.listdir(CHECKPOINTS_DIR):
    last_checkpoint = get_last_checkpoint(CHECKPOINTS_DIR)
    if last_checkpoint:
        print(f"✅ Checkpoint local trouvé — reprise depuis : {last_checkpoint}")
    else:
        print("Dossier de checkpoints présent mais vide/incomplet → entraînement depuis le début.")
else:
    print("Aucun checkpoint trouvé → entraînement depuis le début.")


✅ Checkpoint local trouvé — reprise depuis : /content/drive/MyDrive/fr-fulfulde-mt/checkpoints/checkpoint-3500


## 9. Métriques d'évaluation (BLEU + chrF++)

chrF++ est généralement plus fiable que BLEU pour les langues morphologiquement riches /
peu dotées comme le fulfulde, car il travaille au niveau caractère et tolère mieux les petites
variations orthographiques. On rapporte les deux.


In [ ]:
import evaluate
import numpy as np

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    bleu_result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels_bleu)
    chrf_result = chrf.compute(predictions=decoded_preds, references=decoded_labels_bleu)

    return {
        "bleu": bleu_result["score"],
        "chrf++": chrf_result["score"],
    }


In [ ]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        # ✅ AJOUTÉ : arrêt automatique si eval_loss ne s'améliore pas
        # pendant 5 évaluations consécutives (= 500 steps)
        # → économise du temps GPU, réduit le risque de surapprentissage tardif
        EarlyStoppingCallback(early_stopping_patience=5),
    ],
)

# Reprise automatique si checkpoint disponible, sinon entraînement depuis zéro
trainer.train(resume_from_checkpoint=last_checkpoint)


`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...


Step,Training Loss,Validation Loss,Bleu,Chrf++
100,2.571200,2.299691,8.832182,19.975524


Step,Training Loss,Validation Loss,Bleu,Chrf++
100,2.571200,2.299691,8.832182,19.975524
200,2.192700,2.075189,12.487065,23.120092
300,2.162300,1.940583,15.727844,25.722861
400,2.032700,1.855395,20.131591,40.601814
500,2.026700,1.804930,22.229208,43.484919
600,2.024700,1.766068,23.308780,44.154295
700,1.862400,1.734797,23.604924,44.868857
800,1.825200,1.704662,24.352261,45.201293
900,1.758100,1.679153,25.456504,46.380756
1000,1.846000,1.652853,24.622094,45.829516


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


## 10. Évaluation finale sur le jeu de test (jamais vu pendant l'entraînement)

**Correction appliquée :** l'évaluation est maintenant faite **séparément par direction** :
- `FR → FF` : on mesure la capacité du modèle à traduire *vers* le Fulfulde
- `FF → FR` : on mesure la capacité à traduire *depuis* le Fulfulde

Avant, les deux directions étaient mélangées dans un seul score, masquant d'éventuelles
asymétries de performance (fréquentes en MT pour langues peu dotées).


In [ ]:
# ── Évaluation FR → FF ───────────────────────────────────────────────────────
print("=" * 65)
print("  Évaluation FR → FF  (jeu de test, jamais vu pendant l'entraînement)")
print("=" * 65)
results_fr2ff = trainer.evaluate(
    eval_dataset=test_fr2ff,
    metric_key_prefix="test_fr2ff"
)
print(results_fr2ff)

# ── Évaluation FF → FR ───────────────────────────────────────────────────────
print()
print("=" * 65)
print("  Évaluation FF → FR  (jeu de test, jamais vu pendant l'entraînement)")
print("=" * 65)
results_ff2fr = trainer.evaluate(
    eval_dataset=test_ff2fr,
    metric_key_prefix="test_ff2fr"
)
print(results_ff2fr)

# ── Tableau récapitulatif ─────────────────────────────────────────────────────
bleu_fr2ff = results_fr2ff.get("test_fr2ff_bleu",   float("nan"))
chrf_fr2ff = results_fr2ff.get("test_fr2ff_chrf++", float("nan"))
bleu_ff2fr = results_ff2fr.get("test_ff2fr_bleu",   float("nan"))
chrf_ff2fr = results_ff2fr.get("test_ff2fr_chrf++", float("nan"))

print()
print("┌─────────────────┬────────────┬────────────┐")
print("│ Direction       │    BLEU    │   chrF++   │")
print("├─────────────────┼────────────┼────────────┤")
print(f"│ FR → Fulfulde   │  {bleu_fr2ff:>7.2f}   │  {chrf_fr2ff:>7.2f}   │")
print(f"│ Fulfulde → FR   │  {bleu_ff2fr:>7.2f}   │  {chrf_ff2fr:>7.2f}   │")
print("└─────────────────┴────────────┴────────────┘")


## 11. Sauvegarde du modèle final sur Drive


In [ ]:
import json as _json
import shutil

# ── Sauvegarde du modèle et du tokenizer ─────────────────────────
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"✅ Modèle final sauvegardé dans : {FINAL_MODEL_DIR}")

# ── Sauvegarde des métriques par direction ────────────────────────
all_metrics = {
    "FR_to_Fulfulde": results_fr2ff,
    "Fulfulde_to_FR": results_ff2fr,
}
metrics_path = os.path.join(REPORTS_DIR, "test_metrics.json")
os.makedirs(REPORTS_DIR, exist_ok=True)
with open(metrics_path, "w", encoding="utf-8") as f:
    _json.dump(all_metrics, f, ensure_ascii=False, indent=2)
print(f"✅ Métriques sauvegardées dans : {metrics_path}")
print()
print(_json.dumps(all_metrics, ensure_ascii=False, indent=2))

# ── Téléchargement automatique du modèle final en .zip ───────────
# Le zip est créé dans /content pour pouvoir être téléchargé via
# le panneau "Fichiers" de Colab (icône dossier à gauche) ou
# directement avec files.download() ci-dessous.
ZIP_PATH = "/content/model_final_tardigrade.zip"
shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=os.path.dirname(FINAL_MODEL_DIR),
    base_dir=os.path.basename(FINAL_MODEL_DIR),
)
print(f"\n📦 Archive créée : {ZIP_PATH}")
print("   → Télécharge-la via le panneau Fichiers (icône 📁 à gauche)")
print("   → Ou exécute la cellule suivante pour un téléchargement automatique")


In [ ]:
# Téléchargement direct dans ton navigateur (Colab uniquement)
# Décommente et exécute après la sauvegarde ci-dessus.

# from google.colab import files
# files.download(ZIP_PATH)


## 12. Démo d'inférence

Teste ici quelques phrases. **Fais valider ces traductions par un locuteur natif Adamawa**
avant de tirer des conclusions sur la qualité réelle.


In [ ]:
def translate(text, src_lang, tgt_lang, num_beams=5, max_length=MAX_LENGTH):
    """Traduit un texte avec beam search (num_beams=5, cohérent avec l'évaluation)."""
    model.eval()                              # ✅ désactive dropout pour l'inférence
    tokenizer.src_lang = src_lang
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=max_length
    ).to(device)
    with torch.no_grad():                     # ✅ pas de calcul de gradient
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            num_beams=num_beams,
            max_length=max_length,
        )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0]


# ── Démo FR → Fulfulde ───────────────────────────────────────────────────────
print("── FR → Fulfulde ─────────────────────────────")
examples_fr = [
    "Au commencement, Dieu créa les cieux et la terre.",
    "Aimez-vous les uns les autres.",
    "Ne crains pas, car je suis avec toi.",
]
for ex in examples_fr:
    print(f"FR : {ex}")
    print(f"FF : {translate(ex, SRC_LANG, TGT_LANG)}")
    print()

# ── Démo Fulfulde → FR ───────────────────────────────────────────────────────
print("── Fulfulde → FR ─────────────────────────────")
examples_ff = [
    "Alla woni e mum.",
    "Ndaa inɗe ɓiɓɓe Yaakubu.",
    "Yusufu e deerɗiraaɓe muuɗum fuu maayi.",
    "Isra'iila'en ndanyndanytiri, ɓe ɗon ɓesdo.",
]
for ex in examples_ff:
    print(f"FF : {ex}")
    print(f"FR : {translate(ex, TGT_LANG, SRC_LANG)}")
    print()

In [ ]:
import time

# Boucle pour simuler une activité de calcul en arrière-plan
try:
    while True:
        print("Session maintenue active...", flush=True)
        time.sleep(120)  # Attend 2 minutes avant d'afficher à nouveau
except KeyboardInterrupt:
    print("Maintien d'activité arrêté.")


## Notes finales

- **Limite de dialecte** : `fuv_Latn` = Nigerian Fulfulde dans NLLB-200, pas un code Adamawa
  Cameroun dédié. La validation humaine par un locuteur natif Adamawa (toi, idéalement, ou
  quelqu'un d'autre de la zone) reste l'étape la plus importante avant de communiquer des
  résultats.
- **Limite de domaine** : corpus 100% biblique → bon sur ce registre, à valider séparément
  pour tout usage hors de ce domaine (conversation courante, administratif, etc.).
- **Prochaine étape suggérée** : une fois la faisabilité validée, élargir le corpus avec des
  données hors domaine biblique (actualités, conversation, etc.) pour viser un modèle plus
  général, comme discuté.


In [ ]:
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import evaluate
import numpy as np
import pandas as pd

# --- 1. Charger le modèle et le tokenizer finaux depuis Drive ---
# CHANGEMENT: Chargement depuis le dernier checkpoint au lieu du dossier FINAL_MODEL_DIR
print(f"Chargement du modèle depuis le dernier checkpoint : {last_checkpoint}")
final_tokenizer = AutoTokenizer.from_pretrained(last_checkpoint)
final_model = AutoModelForSeq2SeqLM.from_pretrained(last_checkpoint)
device = "cuda" if torch.cuda.is_available() else "cpu"
final_model.to(device)
print(f"Modèle final chargé sur {device}.")

# --- 2. Préparation des données de test (dévaluation) ---
# Les dataframes de test sont déjà disponibles (test_df)
# On recrée les datasets unidirectionnels avec le tokenizer final

# Assurez-vous que TranslationDataset est défini, si non, le copier ici
# (Copie du code de la classe TranslationDataset pour autonomie si la cellule d'origine n'est pas exécutée)
MAX_LENGTH = 128
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, src_lang, tgt_lang,
                 max_length=MAX_LENGTH, bidirectional=True, label="dataset"):
        self.label = label
        pairs = []
        for _, row in df.iterrows():
            fr = str(row["french_text"]).strip()
            ff = str(row["fulfulde_text"]).strip()
            if not fr or not ff:
                continue
            pairs.append((fr, ff, src_lang, tgt_lang))
            if bidirectional:
                pairs.append((ff, fr, tgt_lang, src_lang))

        if not pairs:
            raise ValueError(f"[{label}] Le DataFrame est vide ou ne contient pas de lignes valides.")

        print(f"[{label}] Pré-tokenisation de {len(pairs)} exemples...")
        self.tokenized = []
        n_truncated = 0
        for src_text, tgt_text, s_lang, t_lang in pairs:
            tokenizer.src_lang = s_lang
            tokenizer.tgt_lang = t_lang
            src_enc = tokenizer(src_text, max_length=max_length, truncation=True)
            tgt_enc = tokenizer(text_target=tgt_text, max_length=max_length, truncation=True)
            if (len(src_enc["input_ids"]) >= max_length or
                    len(tgt_enc["input_ids"]) >= max_length):
                n_truncated += 1
            item = dict(src_enc)
            item["labels"] = tgt_enc["input_ids"]
            self.tokenized.append(item)

        pct = 100 * n_truncated / len(pairs)
        icon = "⚠️ " if pct > 10 else "✅"
        msg  = f" → envisage d'augmenter MAX_LENGTH à 192 ou 256." if pct > 10 else ""
        print(f"{icon} [{label}] {len(pairs)} exemples | Troncations : {n_truncated} ({pct:.1f}%){msg}")

    def __len__(self):
        return len(self.tokenized)

    def __getitem__(self, idx):
        return self.tokenized[idx]

# Variables globales pour les langues source et cible, et le chemin du modèle
SRC_LANG = "fra_Latn"
TGT_LANG = "fuv_Latn"

# Recréer les datasets de test avec le tokenizer final
test_fr2ff_final = TranslationDataset(test_df, final_tokenizer, SRC_LANG, TGT_LANG,
                                bidirectional=False, label="TEST FINAL FR→FF")
test_ff2fr_final = TranslationDataset(test_df, final_tokenizer, TGT_LANG, SRC_LANG,
                                bidirectional=False, label="TEST FINAL FF→FR")

# --- 3. Fonction compute_metrics pour l'évaluation ---
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def compute_metrics_final(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    labels = np.where(labels != -100, labels, final_tokenizer.pad_token_id)

    decoded_preds = final_tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = final_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    bleu_result = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels_bleu)
    chrf_result = chrf.compute(predictions=decoded_preds, references=decoded_labels_bleu)

    return {"bleu": bleu_result["score"], "chrf++": chrf_result["score"]}

# --- 4. Créer un Trainer pour l'évaluation ---
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Les arguments d'entraînement sont nécessaires pour le Trainer, même juste pour l'évaluation
# On peut simplifier pour l'évaluation pure
eval_training_args = Seq2SeqTrainingArguments(
    output_dir="./eval_output", # Un dossier temporaire pour l'évaluation
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=5,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

eval_data_collator = DataCollatorForSeq2Seq(final_tokenizer, model=final_model, padding=True)

evaluator = Seq2SeqTrainer(
    model=final_model,
    args=eval_training_args,
    data_collator=eval_data_collator,
    compute_metrics=compute_metrics_final,
    tokenizer=final_tokenizer # IMPORTANT: passer le tokenizer ici pour generation_config
)

print("\n" + "=" * 65)
print("  DÉVALUATION SUR LES DONNÉES DE TEST (jamais vues)")
print("=" * 65)

# Évaluation FR → FF
print("\n── Évaluation FR → FF ───────────────────────────────────────────────")
results_fr2ff_final = evaluator.evaluate(eval_dataset=test_fr2ff_final, metric_key_prefix="test_fr2ff_final")
print(results_fr2ff_final)

# Évaluation FF → FR
print("\n── Évaluation FF → FR ───────────────────────────────────────────────")
results_ff2fr_final = evaluator.evaluate(eval_dataset=test_ff2fr_final, metric_key_prefix="test_ff2fr_final")
print(results_ff2fr_final)

# Affichage récapitulatif
bleu_fr2ff_final = results_fr2ff_final.get("test_fr2ff_final_bleu", float("nan"))
chrf_fr2ff_final = results_fr2ff_final.get("test_fr2ff_final_chrf++", float("nan"))
bleu_ff2fr_final = results_ff2fr_final.get("test_ff2fr_final_bleu", float("nan"))
chrf_ff2fr_final = results_ff2fr_final.get("test_ff2fr_final_chrf++", float("nan"))

print("\n┌───────────────────────────┬────────────┬────────────┐")
print("│ Direction (Modèle Final)  │    BLEU    │   chrF++   │")
print("├───────────────────────────┼────────────┼────────────┤")
print(f"│ FR → Fulfulde             │  {bleu_fr2ff_final:>7.2f}   │  {chrf_fr2ff_final:>7.2f}   │")
print(f"│ Fulfulde → FR             │  {bleu_ff2fr_final:>7.2f}   │  {chrf_ff2fr_final:>7.2f}   │")
print("└───────────────────────────┴────────────┴────────────┘")

# ── Inférence sur phrases prédéfinies (remplace la boucle interactive) ────────
def translate_final(text, src_lang, tgt_lang, num_beams=5, max_length=MAX_LENGTH):
    final_model.eval()
    final_tokenizer.src_lang = src_lang
    inputs = final_tokenizer(
        text, return_tensors="pt", truncation=True, max_length=max_length
    ).to(device)
    with torch.no_grad():
        generated = final_model.generate(
            **inputs,
            forced_bos_token_id=final_tokenizer.convert_tokens_to_ids(tgt_lang),
            num_beams=num_beams,
            max_length=max_length,
        )
    return final_tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

print("\n── FR → Fulfulde (checkpoint final) ──────────")
for phrase in [
    "Au commencement, Dieu créa les cieux et la terre.",
    "Ne crains pas, car je suis avec toi.",
]:
    print(f"FR : {phrase}")
    print(f"FF : {translate_final(phrase, SRC_LANG, TGT_LANG)}")
    print()

print("── Fulfulde → FR (checkpoint final) ──────────")
for phrase in [
    "Alla woni e mum.",
    "Ndaa inɗe ɓiɓɓe Yaakubu.",
]:
    print(f"FF : {phrase}")
    print(f"FR : {translate_final(phrase, TGT_LANG, SRC_LANG)}")
    print()
print("\n✅ Évaluation finale terminée.")


## Interface Gradio

Le fichier `app.py` à la racine du dépôt contient l'interface Gradio prête à déployer sur [Hugging Face Spaces](https://huggingface.co/spaces).

```bash
# Test local
pip install gradio
python app.py
```

Assure-toi que le modèle est bien publié sous `bonopassale/nllb-fra-fuv-finetuned` avant de lancer le Space.
